# Phase 1 — Demand & Pricing Profile (Step 1.5)

**Scope of this notebook** — the checklist from `solution_plan.md` Step 1.5, in order:

| # | Question | Output |
| --- | --- | --- |
| 1 | Date span of bookings, split by status | settled vs committed occupancy classification |
| 2 | Realised price distribution, cut by city / screen type / block / rotation type / size / market tier | ranked price drivers |
| 3 | Occupancy at screen x block x date grain | the booking-expansion transform (`occupancy_timeline`) |
| 4 | Bundle prevalence and bundle-vs-standalone pricing | evidence for the "bundle is one deal" nuance |
| 5 | Rotation-type mix and price-per-slot linearity | evidence for the non-linear slot-pricing nuance |
| 6 | Lost leads: loss reasons, price-gap distribution, negotiation rounds, stage, lead age | the price-cap calibration signal |

**Exit criterion (from the plan):** findings committed here, and a short list of
**empirically supported** price drivers, ranked by effect size, written to
`docs/data_dictionary.md#price-drivers`.

**As-of date.** Step 1.4 established `2026-08-19` as the dataset's as-of date,
triangulated from four independent signals (last `completed` end, first
`upcoming` start, the `active` span, and ridership actuals). Every split below
reuses that date rather than the wall clock — see [`1.4_inventory_shape.md`](../docs/1.4_inventory_shape.md).

## 0 · Environment

In [1]:
from __future__ import annotations

import sys
from pathlib import Path

import numpy as np
import pandas as pd


def _project_root() -> Path:
    """Locate the repository root whether the kernel starts in / or in notebooks/."""
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "src" / "agentiq").is_dir():
            return candidate
    raise FileNotFoundError("Run this notebook from inside the project tree.")


ROOT = _project_root()
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 160)

In [ ]:
%load_ext autoreload
%autoreload 2

from agentiq.data import DataLake, ProjectPaths

PATHS = ProjectPaths(ROOT).ensure_dirs()
lake = DataLake(PATHS.raw_data, cache_dir=PATHS.cache)
PATHS

In [ ]:
bookings = lake["bookings"]
leads = lake["lost_leads"]
screens = lake["screens"]
dim_slot = lake["dim_slot"]
cities = lake["cities"]

AS_OF = pd.Timestamp("2026-08-19")  # fixed in Step 1.4 — do not derive from wall clock
print(f"bookings: {len(bookings):,} rows   lost_leads: {len(leads):,} rows   as-of: {AS_OF.date()}")

## 1.5.1 · Date span and status split

**Goal.** Confirm the `completed` / `active` / `upcoming` partition is exact and
disjoint around `AS_OF`, and that only `completed` rows are safe training data.

**Why this matters.** Splitting on `start_date` for a train/test split leaks
future information — `booked_date` is the only column safe for time-ordered
validation (per the data dictionary). `active` / `upcoming` rows are *committed
occupancy*, a different input from settled price history.

In [ ]:
status_span = (
    bookings.groupby("booking_status")
    .agg(
        lines=("booking_id", "count"),
        start_min=("start_date", "min"),
        start_max=("start_date", "max"),
        end_min=("end_date", "min"),
        end_max=("end_date", "max"),
    )
    .assign(share=lambda f: (f["lines"] / len(bookings) * 100).round(1))
)
status_span

In [ ]:
# Verify the classification is exact, not a heuristic: the three windows must be disjoint
# and every row must fall on the correct side of AS_OF.
completed_ok = (bookings.loc[bookings.booking_status == "completed", "end_date"] <= AS_OF).all()
active_ok = (
    (bookings.loc[bookings.booking_status == "active", "start_date"] <= AS_OF)
    & (bookings.loc[bookings.booking_status == "active", "end_date"] >= AS_OF)
).all()
upcoming_ok = (bookings.loc[bookings.booking_status == "upcoming", "start_date"] > AS_OF).all()

print(f"completed all end <= as-of : {completed_ok}")
print(f"active all span as-of     : {active_ok}")
print(f"upcoming all start > as-of: {upcoming_ok}")

settled = bookings.loc[bookings.booking_status == "completed"].copy()
committed = bookings.loc[bookings.booking_status.isin(["active", "upcoming"])].copy()
print(f"\nsettled (training data): {len(settled):,} lines")
print(f"committed (occupancy):   {len(committed):,} lines")

## 1.5.2 · Realised price distribution by facet

**Goal.** Which cuts actually move `contracted_price_per_slot_per_day`? This
drives the Phase 6 pricing feature set — only report cuts with a real,
measured effect size, not every column that happens to exist.

Uses `settled` only (completed bookings) so the distribution reflects realised
market prices, never committed-but-unrealised occupancy.

In [ ]:
PRICE_COL = "contracted_price_per_slot_per_day"

settled["screen_type"] = settled["screen_id"].map(screens.set_index("screen_id")["screen_type"])
settled["screen_size"] = settled["screen_id"].map(screens.set_index("screen_id")["screen_size"])
settled["market_tier"] = settled["city_id"].map(cities.set_index("city_id")["market_tier"])


def price_by(facet: str) -> pd.DataFrame:
    g = settled.groupby(facet)[PRICE_COL]
    out = g.agg(lines="count", median_price="median", mean_price="mean", std_price="std").sort_values(
        "median_price", ascending=False
    )
    out["cv"] = (out["std_price"] / out["mean_price"]).round(3)
    return out.round(2)


for facet in ["city_id", "market_tier", "screen_type", "screen_size", "time_block_id", "rotation_type"]:
    print(f"\n--- price by {facet} ---")
    print(price_by(facet))

In [ ]:
# Rank facets by effect size: spread of median price across the facet's own levels,
# normalised by the overall median so city-tier and slot-count are comparable.
overall_median = settled[PRICE_COL].median()
effect_sizes = {}
for facet in ["city_id", "market_tier", "screen_type", "screen_size", "time_block_id", "rotation_type"]:
    medians = settled.groupby(facet)[PRICE_COL].median()
    effect_sizes[facet] = ((medians.max() - medians.min()) / overall_median).round(3)

ranked_drivers = (
    pd.Series(effect_sizes, name="relative_spread")
    .sort_values(ascending=False)
    .to_frame()
)
ranked_drivers

## 1.5.3 · Occupancy — the booking-expansion transform

**Goal.** For each `screen x time_block x date`, how many of the (at most 6)
rotation slots are claimed? A booking line only states `start_date` /
`end_date`; it must be **expanded** across every date in that range to answer
an occupancy question. This expansion is a core reusable artifact — Step 1.4
already used a sweep-line version of it to prove the 6-slot ceiling; here it's
built explicitly and reused for the price/occupancy join.

This is deliberately done as a **sweep line over start/end deltas**, not a
per-day explode, so it stays cheap at 191k lines x up to 180 days each.

In [ ]:
def occupancy_timeline(lines: pd.DataFrame) -> pd.DataFrame:
    """Daily claimed-slot count per (screen_id, time_block_id), via a +/- sweep line.

    Returns one row per (screen_id, time_block_id, date) with non-zero occupancy.
    Dates with zero claimed slots are simply absent — sparse by construction.
    """
    starts = lines[["screen_id", "time_block_id", "start_date", "slots_booked_per_day"]].rename(
        columns={"start_date": "date", "slots_booked_per_day": "delta"}
    )
    ends = lines[["screen_id", "time_block_id", "end_date", "slots_booked_per_day"]].rename(
        columns={"end_date": "date", "slots_booked_per_day": "delta"}
    )
    ends["date"] = ends["date"] + pd.Timedelta(days=1)
    ends["delta"] = -ends["delta"]

    events = pd.concat([starts, ends], ignore_index=True)
    events = events.groupby(["screen_id", "time_block_id", "date"], as_index=False)["delta"].sum()
    events = events.sort_values(["screen_id", "time_block_id", "date"])

    events["occupied_slots"] = events.groupby(["screen_id", "time_block_id"])["delta"].cumsum()
    return events.loc[events["occupied_slots"] > 0, ["screen_id", "time_block_id", "date", "occupied_slots"]]


occupancy = occupancy_timeline(bookings)
print(f"{len(occupancy):,} non-zero (screen, block, date) occupancy rows")
print(f"peak occupied_slots observed: {occupancy['occupied_slots'].max()}  (capacity ceiling is 6, per Step 1.4)")
assert occupancy["occupied_slots"].max() <= 6, "occupancy exceeds the measured capacity ceiling"
occupancy.head()

In [ ]:
# Occupancy as of AS_OF only, so this is comparable to Step 1.4's headline utilisation figures.
occ_as_of = occupancy.loc[occupancy["date"] == AS_OF]
print(f"screen x block pairs occupied on the as-of date: {len(occ_as_of):,}")
occ_as_of["occupied_slots"].describe()

## 1.5.4 · Bundle prevalence and bundle-vs-standalone pricing

**Goal.** Evidence for the "a bundle is one deal, not N deals" nuance. Per the
data dictionary, 135,624 lines (71%) belong to only 1,277 bundle deals
(~106 lines/bundle), while 55,485 lines are one deal each. `deal_total_value`
is repeated on every line of a deal, so it must be de-duplicated by `deal_id`
before any revenue sum — summing it naively overstates bundle revenue ~106x.

In [ ]:
bundle_share = (
    bookings.groupby("is_bundle")
    .agg(lines=("booking_id", "count"), deals=("deal_id", "nunique"))
    .assign(lines_per_deal=lambda f: (f["lines"] / f["deals"]).round(1))
)
bundle_share

In [ ]:
# Correct, de-duplicated deal-level value: one row per deal, not per line.
deal_value = bookings.drop_duplicates("deal_id")[["deal_id", "is_bundle", "deal_total_value"]]
naive_sum = bookings["deal_total_value"].sum()
correct_sum = deal_value["deal_total_value"].sum()
print(f"naive line-level sum of deal_total_value:  {naive_sum:,.0f}")
print(f"correct deal-level sum (post de-dup):      {correct_sum:,.0f}")
print(f"overstatement factor:                      {naive_sum / correct_sum:.1f}x")

In [ ]:
# Does a bundled line price differently from a standalone line, at the per-slot-per-day grain
# (which already normalises out the volume difference)?
bundle_price = (
    settled.groupby("is_bundle")[PRICE_COL]
    .agg(lines="count", median_price="median", mean_price="mean")
    .round(2)
)
bundle_price

## 1.5.5 · Rotation-type mix and price-per-slot linearity

**Goal.** Test explicitly whether price per slot scales linearly with slots
booked — the non-linearity nuance the problem statement names, and the input
to Phase 6's base-rate model and Phase 3's attention-curve model.

Step 1.4 already found this pattern on the full booking set; here it is
re-tested on **settled bookings only**, and then checked for confounding with
`screen_type` and `time_block_id` — the explicit follow-up the plan calls for.

In [ ]:
linearity = (
    settled.groupby("slots_booked_per_day")[PRICE_COL]
    .agg(lines="count", median_price_per_slot="median")
    .round(2)
)
linearity["vs_single_slot_pct"] = (
    (linearity["median_price_per_slot"] / linearity.loc[1, "median_price_per_slot"] - 1) * 100
).round(1)
linearity

In [ ]:
# Confound check: does the discount survive within a single (screen_type, time_block_id) cell,
# or is it an artefact of which screen types / blocks tend to be bought in bulk?
within_cell = (
    settled.groupby(["screen_type", "time_block_id", "slots_booked_per_day"])[PRICE_COL]
    .median()
    .reset_index()
)


def slope_within_cell(group: pd.DataFrame) -> float | None:
    if group["slots_booked_per_day"].nunique() < 2:
        return None
    return np.polyfit(group["slots_booked_per_day"], group[PRICE_COL], 1)[0]


cell_slopes = (
    within_cell.groupby(["screen_type", "time_block_id"])
    .apply(slope_within_cell, include_groups=False)
    .dropna()
    .rename("price_per_slot_vs_slot_count_slope")
)
print(f"cells with a fittable slope: {len(cell_slopes)}")
print(f"share of cells with a negative slope (price/slot falls as slots rise): "
      f"{(cell_slopes < 0).mean():.1%}")
cell_slopes.describe()

**Read on the confound check:** if the negative-slope share above is high
(most `screen_type x time_block_id` cells individually show falling
price-per-slot as slot count rises), the discount is a real volume effect, not
an artefact of block/type mix. If it is closer to 50%, the network-level effect
in §1.5.5's first table is confounded and Phase 6 must control for
`screen_type`/`time_block_id` explicitly rather than fitting slot count alone.

## 1.5.6 · Lost leads — the price-cap calibration signal

**Goal.** Quantify at what price gap deals demonstrably die, so Phase 6's price
cap has empirical backing rather than being an arbitrary guardrail.
`price_gap_pct = (quote - client_target) / client_target`; it and
`client_target_price_per_slot_per_day` are null on the 531 `initial_inquiry`
leads — the lead died before a quote or counter-offer existed, so absence here
is meaningful, not missing data.

In [ ]:
loss_by_stage = (
    leads.groupby("sales_stage_reached")
    .agg(
        leads=("lead_id", "count"),
        has_quote=("quoted_price_per_slot_per_day", lambda s: s.notna().sum()),
        median_gap_pct=("price_gap_pct", "median"),
        median_rounds=("negotiation_rounds", "median"),
    )
)
loss_by_stage

In [ ]:
priced_leads = leads.dropna(subset=["price_gap_pct"]).copy()

# Bucket the price gap and read off win-adjacent behaviour (stage reached, rounds fought)
# as a proxy for "how close was this deal to landing" at each gap level.
priced_leads["gap_bucket"] = pd.cut(
    priced_leads["price_gap_pct"],
    bins=[-1, 0, 0.05, 0.10, 0.15, 0.20, 1],
    labels=["<=0%", "0-5%", "5-10%", "10-15%", "15-20%", ">20%"],
)

gap_profile = priced_leads.groupby("gap_bucket").agg(
    leads=("lead_id", "count"),
    reached_verbal_or_later=(
        "sales_stage_reached",
        lambda s: s.isin(["verbal_agreement", "contract_sent"]).mean(),
    ),
    median_rounds=("negotiation_rounds", "median"),
)
gap_profile["reached_verbal_or_later"] = (gap_profile["reached_verbal_or_later"] * 100).round(1)
gap_profile

In [ ]:
loss_reasons = (
    leads["loss_reason_detail"]
    .value_counts()
    .head(10)
    .to_frame("leads")
    .assign(share_pct=lambda f: (f["leads"] / len(leads) * 100).round(1))
)
loss_reasons

In [ ]:
# Lead age at time of loss, and how age relates to how far the deal got —
# the recency-decay input for Phase 6's pipeline-pressure signal.
leads["cycle_days"] = (leads["lost_date"] - leads["lead_date"]).dt.days
leads["age_at_as_of"] = (AS_OF - leads["lead_date"]).dt.days

cycle_by_stage = leads.groupby("sales_stage_reached")["cycle_days"].median().sort_values()
print("median days from lead to loss, by stage reached:")
cycle_by_stage

In [ ]:
competitor_effect = (
    leads.groupby("competitor_mentioned")
    .agg(
        leads=("lead_id", "count"),
        median_gap_pct=("price_gap_pct", "median"),
        median_rounds=("negotiation_rounds", "median"),
    )
)
competitor_effect

## 1.5.7 · Summary — price drivers ranked by effect size

**Exit criterion.** This table is the notebook's deliverable per the plan: an
empirically ranked list of price drivers, to be committed to
`docs/data_dictionary.md#price-drivers` once reviewed at the Step 1.9 gate.

In [ ]:
ranked_drivers

## Carry-forward

| Output | Consumed by |
| --- | --- |
| Settled / committed split (§1.5.1) | Every train/test split downstream |
| Ranked price drivers (§1.5.2, §1.5.7) | Phase 6 base-rate feature set |
| `occupancy_timeline()` (§1.5.3) | Step 6.1 scarcity signal, Phase 7 availability constraint |
| Bundle de-duplication rule (§1.5.4) | Phase 7 joint bundle pricing |
| Slot-price linearity + confound check (§1.5.5) | Phase 6 base-rate model, Phase 3 attention curve |
| Price-gap / stage / competitor evidence (§1.5.6) | Phase 6 price-cap calibration, win-probability model |

**Next — Step 1.6 (context profile):** POI footfall scale and walking-radius
validation, event surge characteristics, zone demographic discriminants, and
the normalised daypart ridership curve per route/corridor.